# Constructor Rank API Discovery

This notebook explores public APIs to fetch **constructor standings before an event**.

Goal:
- Given `(season, round_number)`, return constructor standings **before** that event.
- For round `R`, query standings at round `R-1` (same season).

Primary candidate:
- Ergast/Jolpica-style endpoint: `/{season}/{round}/constructorStandings.json`

In [1]:
import requests
import pandas as pd
from typing import Dict, List, Tuple

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

In [2]:
# Candidate base URLs for Ergast-compatible API providers.
# Keep order from most likely/current to legacy mirrors.
BASE_URLS = [
    "https://api.jolpi.ca/ergast/f1",     # Jolpica (Ergast-compatible)
    "https://ergast.com/api/f1",          # legacy Ergast URL (may be unavailable)
]


def _get_json(url: str, timeout: int = 20):
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()


def fetch_constructor_standings_at_round(season: int, round_number: int) -> Tuple[str, List[dict]]:
    """Return (base_url_used, standings_list) for a season/round.

    standings_list is the Ergast/Jolpica ConstructorStandings payload rows.
    """
    if round_number < 1:
        raise ValueError("round_number must be >= 1")

    last_err = None
    for base in BASE_URLS:
        url = f"{base}/{season}/{round_number}/constructorStandings.json"
        try:
            data = _get_json(url)
            lists = data.get("MRData", {}).get("StandingsTable", {}).get("StandingsLists", [])
            if not lists:
                continue
            rows = lists[0].get("ConstructorStandings", [])
            if rows:
                return base, rows
        except Exception as exc:
            last_err = exc
            continue

    if last_err is not None:
        raise RuntimeError(f"No standings data found. Last error: {last_err}")
    raise RuntimeError("No standings data found from candidate APIs.")


def standings_to_df(rows: List[dict]) -> pd.DataFrame:
    out = []
    for r in rows:
        c = r.get("Constructor", {})
        out.append(
            {
                "position": int(r.get("position", 999)),
                "points": float(r.get("points", 0.0)),
                "wins": int(r.get("wins", 0)),
                "constructor_id": c.get("constructorId"),
                "constructor_name": c.get("name"),
                "constructor_nationality": c.get("nationality"),
            }
        )
    return pd.DataFrame(out).sort_values("position").reset_index(drop=True)

In [3]:
def fetch_constructor_ranks_before_event(season: int, event_round: int) -> Tuple[str, int, pd.DataFrame]:
    """Fetch standings before event_round.

    If event_round == 1, there is no same-season prior round.
    In that case this function raises a ValueError and you can choose
    a fallback policy (e.g., previous season final standings).
    """
    if event_round <= 1:
        raise ValueError(
            "No prior same-season standings exist before round 1. "
            "Use previous season final standings as fallback."
        )

    prior_round = event_round - 1
    base_used, rows = fetch_constructor_standings_at_round(season, prior_round)
    df = standings_to_df(rows)
    return base_used, prior_round, df

In [4]:
# Demo: standings BEFORE round 10 of 2024 (i.e., after round 9)
SEASON = 2024
EVENT_ROUND = 10

base, prior_round, standings_df = fetch_constructor_ranks_before_event(SEASON, EVENT_ROUND)
print(f"API base used: {base}")
print(f"Queried standings at season={SEASON}, round={prior_round} (before event round {EVENT_ROUND})")
standings_df.head(10)

API base used: https://api.jolpi.ca/ergast/f1
Queried standings at season=2024, round=9 (before event round 10)


,position,points,wins,constructor_id,constructor_name,constructor_nationality
0,1,301.0,6,red_bull,Red Bull,Austrian
1,2,252.0,2,ferrari,Ferrari,Italian
2,3,212.0,1,mclaren,McLaren,British
3,4,124.0,0,mercedes,Mercedes,German
4,5,58.0,0,aston_martin,Aston Martin,British
5,6,28.0,0,rb,RB F1 Team,Italian
6,7,7.0,0,haas,Haas F1 Team,American
7,8,5.0,0,alpine,Alpine F1 Team,French
8,9,2.0,0,williams,Williams,British
9,10,0.0,0,sauber,Sauber,Swiss


In [5]:
# Optional helper: map constructor names to rank for pipeline use
name_to_rank = dict(zip(standings_df["constructor_name"], standings_df["position"]))
name_to_rank

{'Red Bull': 1,
 'Ferrari': 2,
 'McLaren': 3,
 'Mercedes': 4,
 'Aston Martin': 5,
 'RB F1 Team': 6,
 'Haas F1 Team': 7,
 'Alpine F1 Team': 8,
 'Williams': 9,
 'Sauber': 10}

## Additional API candidates for 2025 coverage

Besides Ergast/Jolpica round standings, probe:

- `f1api.dev` season constructor standings snapshot
- potential `f1api.dev` round endpoint variant (if supported)

Also use `FastF1` schedule to test coverage across the 2025 rounds.

In [6]:
import fastf1


def get_2025_rounds_from_fastf1() -> list[int]:
    sched = fastf1.get_event_schedule(2025)
    rounds = (
        sched["RoundNumber"].dropna().astype(int)
        .loc[lambda s: s > 0]
        .sort_values()
        .unique()
        .tolist()
    )
    return rounds


def probe_jolpica_round(year: int, rnd: int) -> dict:
    url = f"https://api.jolpi.ca/ergast/f1/{year}/{rnd}/constructorStandings.json"
    try:
        data = requests.get(url, timeout=20).json()
        rows = (
            data.get("MRData", {})
            .get("StandingsTable", {})
            .get("StandingsLists", [])
        )
        n = len(rows[0].get("ConstructorStandings", [])) if rows else 0
        return {"ok": n > 0, "count": n, "url": url, "error": None}
    except Exception as exc:  # noqa: BLE001
        return {"ok": False, "count": 0, "url": url, "error": str(exc)}


def probe_f1api_season(year: int) -> dict:
    url = f"https://f1api.dev/api/{year}/constructors-championship"
    try:
        data = requests.get(url, timeout=20).json()
        rows = data.get("constructors_championship", [])
        return {"ok": len(rows) > 0, "count": len(rows), "url": url, "error": None}
    except Exception as exc:  # noqa: BLE001
        return {"ok": False, "count": 0, "url": url, "error": str(exc)}


def probe_f1api_round_variant(year: int, rnd: int) -> dict:
    # Experimental guess endpoint (may not exist)
    url = f"https://f1api.dev/api/{year}/{rnd}/constructors-championship"
    try:
        r = requests.get(url, timeout=20)
        if r.status_code != 200:
            return {"ok": False, "count": 0, "url": url, "error": f"HTTP {r.status_code}"}
        data = r.json()
        rows = data.get("constructors_championship", [])
        return {"ok": len(rows) > 0, "count": len(rows), "url": url, "error": None}
    except Exception as exc:  # noqa: BLE001
        return {"ok": False, "count": 0, "url": url, "error": str(exc)}

In [7]:
rounds_2025 = get_2025_rounds_from_fastf1()
print(f"Detected 2025 rounds from FastF1: {len(rounds_2025)}")
print(rounds_2025)

records = []
for rnd in rounds_2025:
    jol = probe_jolpica_round(2025, rnd)
    f1r = probe_f1api_round_variant(2025, rnd)
    records.append(
        {
            "round": rnd,
            "jolpica_ok": jol["ok"],
            "jolpica_count": jol["count"],
            "f1api_round_ok": f1r["ok"],
            "f1api_round_count": f1r["count"],
            "f1api_round_error": f1r["error"],
        }
    )

coverage_df = pd.DataFrame(records)
coverage_df

req         WARNING 	DEFAULT CACHE ENABLED! (9.78 GB) /Users/aminnami/Library/Caches/fastf1


Detected 2025 rounds from FastF1: 24
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]


,round,jolpica_ok,jolpica_count,f1api_round_ok,f1api_round_count,f1api_round_error
0,1,True,10,False,0,HTTP 404
1,2,True,10,False,0,HTTP 404
2,3,True,10,False,0,HTTP 404
3,4,True,10,False,0,HTTP 404
4,5,True,10,False,0,HTTP 404
5,6,True,10,False,0,HTTP 404
6,7,True,10,False,0,HTTP 404
7,8,True,10,False,0,HTTP 404
8,9,True,10,False,0,HTTP 404
9,10,True,10,False,0,HTTP 404


In [8]:
season_probe = probe_f1api_season(2025)

summary = pd.DataFrame(
    [
        {
            "source": "jolpica_round_endpoint",
            "available_rounds": int(coverage_df["jolpica_ok"].sum()),
            "total_rounds": len(coverage_df),
            "coverage_pct": float(coverage_df["jolpica_ok"].mean() * 100.0),
        },
        {
            "source": "f1api_round_variant",
            "available_rounds": int(coverage_df["f1api_round_ok"].sum()),
            "total_rounds": len(coverage_df),
            "coverage_pct": float(coverage_df["f1api_round_ok"].mean() * 100.0),
        },
        {
            "source": "f1api_season_snapshot",
            "available_rounds": int(season_probe["ok"]),
            "total_rounds": 1,
            "coverage_pct": 100.0 if season_probe["ok"] else 0.0,
        },
    ]
)

print("Coverage summary")
print(summary.to_string(index=False))
print("\nSeason snapshot details:")
print(season_probe)

print("\nRounds where Jolpica is missing in 2025:")
print(coverage_df.loc[~coverage_df["jolpica_ok"], ["round", "jolpica_count"]].to_string(index=False))

Coverage summary
                source  available_rounds  total_rounds  coverage_pct
jolpica_round_endpoint                24            24         100.0
   f1api_round_variant                 0            24           0.0
 f1api_season_snapshot                 1             1         100.0

Season snapshot details:
{'ok': True, 'count': 10, 'url': 'https://f1api.dev/api/2025/constructors-championship', 'error': None}

Rounds where Jolpica is missing in 2025:
Empty DataFrame
Columns: [round, jolpica_count]
Index: []


## Practical recommendation

If 2025 round-level coverage is incomplete:

1. Keep `Jolpica/Ergast` as primary source for round-specific pre-event standings.
2. For rounds with missing data, fallback to the latest prior available round in same season.
3. If none exists (e.g. round 1), fallback to configured default (or previous season final if policy changes).
4. Use `f1api.dev` season snapshot only as a coarse backup when round granularity is unavailable.